# Notebook 51: The Territorial Curriculum Catalog

In this notebook, we unify our data to create the Master Territorial Curriculum Catalog.
We map `Tripartite Macro-Track` -> `Sub-Track` -> `Subjects` -> `Territorial Availability (Region, Province, Municipality)`.
This hierarchical structure is essential for the Phase 2 UI.


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

ROOT = Path('c:/Users/Dell/Documents/Antigravity/Italienation').resolve()
PROC = ROOT / 'local_data/processed'

# Load demographic and school data
students = pd.read_parquet(ROOT / 'local_data/UNICA/hf_students_upper_sec_stat_2024_25.parquet')
scuole = pd.read_parquet(ROOT / 'local_data/Scuola_in_chiaro/scuole/SCUANAGRAFESTAT.parquet')

# Clean Tracks
students['TIPOPERCORSO'] = students['TIPOPERCORSO'].fillna('Sconosciuto').str.title()
students['INDIRIZZO'] = students['INDIRIZZO'].fillna('Sconosciuto').str.title()

# Join with geography
df = pd.merge(students[['CODICESCUOLA', 'TIPOPERCORSO', 'INDIRIZZO']], 
              scuole[['CODICESCUOLA', 'REGIONE', 'PROVINCIA', 'DESCRIZIONECOMUNE']], 
              on='CODICESCUOLA', how='inner')

# Load Subjects (Adozioni)
textbook_dir = ROOT / 'local_data/Scuola_in_chiaro/adozioni_libri_di_testo'
parquet_files = list(textbook_dir.glob('*.parquet'))

df_list = []
for f in parquet_files:
    df_temp = pd.read_parquet(f, columns=['CODICESCUOLA', 'DISCIPLINA'])
    df_temp = df_temp.dropna().drop_duplicates()
    df_temp['DISCIPLINA'] = df_temp['DISCIPLINA'].str.title()
    df_list.append(df_temp)

adozioni = pd.concat(df_list, ignore_index=True).drop_duplicates()

# Group subjects per school
school_subjects = adozioni.groupby('CODICESCUOLA')['DISCIPLINA'].apply(lambda x: list(set(x))).reset_index()

# Merge subjects into our main dataframe
df = pd.merge(df, school_subjects, on='CODICESCUOLA', how='left')
df['DISCIPLINA'] = df['DISCIPLINA'].apply(lambda x: x if isinstance(x, list) else [])

print(f"Data merged. Total school records: {len(df)}")


Data merged. Total school records: 62692


### Build the Hierarchical JSON Catalog
The structure will be:
`Catalog[Region][Province][Municipality][MacroTrack][SubTrack] = [List of Subjects]`


In [2]:
catalog = {}

for row in df.itertuples(index=False):
    reg = str(row.REGIONE).title()
    prov = str(row.PROVINCIA).title()
    com = str(row.DESCRIZIONECOMUNE).title()
    tipo = str(row.TIPOPERCORSO)
    ind = str(row.INDIRIZZO)
    mat = row.DISCIPLINA
    
    if reg not in catalog:
        catalog[reg] = {}
    if prov not in catalog[reg]:
        catalog[reg][prov] = {}
    if com not in catalog[reg][prov]:
        catalog[reg][prov][com] = {}
    if tipo not in catalog[reg][prov][com]:
        catalog[reg][prov][com][tipo] = {}
    
    if ind not in catalog[reg][prov][com][tipo]:
        catalog[reg][prov][com][tipo][ind] = set(mat)
    else:
        catalog[reg][prov][com][tipo][ind].update(mat)

# Convert sets to sorted lists
for reg in catalog:
    for prov in catalog[reg]:
        for com in catalog[reg][prov]:
            for tipo in catalog[reg][prov][com]:
                for ind in catalog[reg][prov][com][tipo]:
                    catalog[reg][prov][com][tipo][ind] = sorted(list(catalog[reg][prov][com][tipo][ind]))

# Export to JSON
json_path = PROC / 'master_territorial_curriculum.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(catalog, f, ensure_ascii=False, indent=2)

print(f"Hierarchical JSON exported to {json_path}")


Hierarchical JSON exported to C:\Users\Dell\Documents\Antigravity\Italienation\local_data\processed\master_territorial_curriculum.json


### Build the Flat CSV Catalog
For traditional querying and database loading.


In [3]:
# Explode the subjects so we have one row per (School, Subject) or rather (Municipality, Track, Subject)
df_exploded = df.explode('DISCIPLINA').dropna(subset=['DISCIPLINA'])
df_flat = df_exploded[['REGIONE', 'PROVINCIA', 'DESCRIZIONECOMUNE', 'TIPOPERCORSO', 'INDIRIZZO', 'DISCIPLINA']].drop_duplicates()

# Format clean up
df_flat['REGIONE'] = df_flat['REGIONE'].str.title()
df_flat['PROVINCIA'] = df_flat['PROVINCIA'].str.title()
df_flat['DESCRIZIONECOMUNE'] = df_flat['DESCRIZIONECOMUNE'].str.title()

csv_path = PROC / 'master_territorial_curriculum.csv'
df_flat.to_csv(csv_path, index=False)

print(f"Flat CSV exported to {csv_path}")
display(df_flat.head())


Flat CSV exported to C:\Users\Dell\Documents\Antigravity\Italienation\local_data\processed\master_territorial_curriculum.csv


,REGIONE,PROVINCIA,DESCRIZIONECOMUNE,TIPOPERCORSO,INDIRIZZO,DISCIPLINA
5,Lombardia,Brescia,Breno,Liceo,Artistico Nuovo Ordinamento - Biennio Comune,Religione Cattolica/Attivita' Alternativa
5,Lombardia,Brescia,Breno,Liceo,Artistico Nuovo Ordinamento - Biennio Comune,"Scienze Naturali (Biologia, Chimica, Scienze D..."
5,Lombardia,Brescia,Breno,Liceo,Artistico Nuovo Ordinamento - Biennio Comune,Matematica
5,Lombardia,Brescia,Breno,Liceo,Artistico Nuovo Ordinamento - Biennio Comune,Scienze Umane
5,Lombardia,Brescia,Breno,Liceo,Artistico Nuovo Ordinamento - Biennio Comune,Storia Dell'Arte
